# [Feature Update (2026-07-14)] CREATE OR ALTER is generally available for 24 object types — 検証ノートブック

## このノートブックについて

Zenn 記事「[[Feature Update (2026-07-14)] CREATE OR ALTER is generally available for 24 object types](https://zenn.dev/gtk0326/articles/i165-feature-update-2026-07-14-create-or-alter-is-)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: TABLE を作成し、再実行でカラムを追加する

まず 2 カラムのテーブルを作成し、データを入れておきます。


In [ ]:
CREATE OR ALTER TABLE customers (
  id   NUMBER,
  name STRING
);

INSERT INTO customers VALUES (1, 'Alice'), (2, 'Bob');

CREATE OR ALTER TABLE customers (
  id    NUMBER,
  name  STRING,
  email STRING
);

SELECT * FROM customers ORDER BY id;

## ステップ2: WAREHOUSE のサイズを再実行で変更する

アカウントレベルのオブジェクトも同じパターンで管理できます。


In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR ALTER WAREHOUSE COA_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND   = 60
  AUTO_RESUME    = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE OR ALTER WAREHOUSE COA_WH
  WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND   = 120
  AUTO_RESUME    = TRUE
  INITIALLY_SUSPENDED = TRUE;

SHOW WAREHOUSES LIKE 'COA_WH';
SELECT "name", "size", "auto_suspend" FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

CREATE OR ALTER TABLE customers (
  id            NUMBER,
  customer_name STRING,
  email         STRING
);

-- email にはデータが入っている
SELECT * FROM customers ORDER BY id;
-- 1  Alice  alice@example.com
-- 2  Bob    bob@example.com

CREATE OR ALTER TABLE customers (
  id           NUMBER,
  name         STRING,
  mail_address STRING
);

SELECT * FROM customers ORDER BY id;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。